# Lymphocyte subclustering

Leiden subclustering of the `manual_celltype == "Lymphocytes"` cells, with phenotype
signatures taken from the `cd8_subclustering_long` sheet of
`BRAIN_CLUSTERING_GUIDE_IRENE_Vs.xlsx`.

**Read the caveats at the bottom before using the labels.** There are 429 lymphocytes
with a median of 162 transcripts each, which supports 3 subclusters - not the 20+
CD8 states the sheet describes.

## 1. Load the object

`X` is normalised + log1p, which is what `score_genes` and the dotplots need.
`layers['counts']` holds the raw counts.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt

sc.settings.verbosity = 1

H5AD_PATH = Path(
    "/Users/valishashah/Library/CloudStorage/Box-Box/Kaech Lab Folder/Valisha/"
    "AD Serial Infection Project  (Irene & Brian W)/Spatial Transcriptomics 20260518/"
    "RESULTS/20240627__192310__KAECH_AD_GBM_240627/08_Plaque_Proximity_Analysis/"
    "adata_combined_with_alphashape_plaque_edge_15_20_30_40um.h5ad"
)

MARKER_XLSX_PATH = Path(
    "/Users/valishashah/Library/CloudStorage/Box-Box/Kaech Lab Folder/Valisha/"
    "Gene List of sub-clustering/BRAIN_CLUSTERING_GUIDE_IRENE_Vs.xlsx"
)
MARKER_SHEET = "cd8_subclustering_long"

OUTPUT_DIR = Path(
    "/Users/valishashah/Library/CloudStorage/Box-Box/Kaech Lab Folder/Valisha/"
    "AD Serial Infection Project  (Irene & Brian W)/Spatial Transcriptomics 20260518/"
    "RESULTS/20240627__192310__KAECH_AD_GBM_240627/10_Lymphocyte_Subclustering"
)
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

adata = ad.read_h5ad(H5AD_PATH)

# X is already normalised + log1p (uns["log1p"] is set and the values are not
# integers); layers["counts"] holds the raw counts.
print(adata)
print()
print("X is log1p-normalised:", "log1p" in adata.uns)
print("raw counts layer:", list(adata.layers))
print("latent space:", adata.obsm["X_scVI"].shape)

## 2. Phenotype signatures from the Excel sheet

The sheet is long format - one row per gene plus the phenotype it belongs to. Three
things have to be handled: genes listed twice, genes the Xenium panel does not carry,
and the handful of Trm genes the sheet marks as *down* rather than up.

The sheet only describes CD8 states, so lineage-check signatures are added. Without
them a B-cell subcluster gets labelled with the nearest available CD8 state.

In [ ]:
# Phenotype signatures from the Excel sheet. The sheet is long format:
# one row per gene, with the phenotype it belongs to.
MINIMUM_GENES_PER_SIGNATURE = 2

marker_sheet = pd.read_excel(MARKER_XLSX_PATH, sheet_name=MARKER_SHEET)
marker_sheet = marker_sheet.rename(columns={"Unnamed: 2": "note"})
marker_sheet["Gene"] = marker_sheet["Gene"].astype(str).str.strip()
marker_sheet["Phenotype"] = marker_sheet["Phenotype"].astype(str).str.strip()

# The sheet marks a few Trm genes as "down"; those must not be scored as if
# high expression supported the phenotype.
down_genes = marker_sheet["note"].astype(str).str.lower().eq("down")
print("genes flagged 'down' in the sheet (excluded from scoring):")
print(marker_sheet.loc[down_genes, ["Gene", "Phenotype"]].to_string(index=False))
marker_sheet = marker_sheet.loc[~down_genes]

panel_genes = set(adata.var_names)
marker_sheet["in_panel"] = marker_sheet["Gene"].isin(panel_genes)

# Deduplicate: some genes are listed twice within a phenotype.
signature_genes = {}
for phenotype, group in marker_sheet.groupby("Phenotype", sort=False):
    present = sorted(set(group.loc[group["in_panel"], "Gene"]))
    signature_genes[phenotype] = present

coverage = pd.DataFrame(
    {
        "genes_in_sheet": marker_sheet.groupby("Phenotype", sort=False)["Gene"].nunique(),
        "genes_in_panel": {k: len(v) for k, v in signature_genes.items()},
    }
)
coverage["genes"] = [", ".join(signature_genes[p]) for p in coverage.index]
coverage["scored"] = coverage["genes_in_panel"] >= MINIMUM_GENES_PER_SIGNATURE

SCORED_SIGNATURES = {
    phenotype: genes
    for phenotype, genes in signature_genes.items()
    if len(genes) >= MINIMUM_GENES_PER_SIGNATURE
}

# The sheet is a CD8 sheet: it has no B-cell, myeloid or neuronal signature, so
# on its own it would give a B-cell subcluster the nearest CD8 label available
# ("Naive like", via Sell / Ccr7 / Cxcr5). These lineage checks are added so a
# subcluster that is not a CD8 state shows up as one instead of being mislabelled.
# Edit freely - they are not from the workbook.
LINEAGE_CHECK_SIGNATURES = {
    "CHECK: T cell lineage": ["Cd3e", "Cd8a", "Cd8b1", "Trbc1", "Cd2", "Thy1"],
    "CHECK: B cell lineage": ["Cd19", "Ms4a1", "Ighd", "Ighm", "Cd74", "H2-Ab1"],
    "CHECK: myeloid lineage": ["Csf1r", "Itgam", "Aif1", "C1qa", "P2ry12"],
    "CHECK: neuronal (ambient/doublet)": [
        "Chgb", "Tubb3", "Dclk1", "Ncam1", "Gpm6b", "Rbfox3",
    ],
}
for check_name, check_genes in LINEAGE_CHECK_SIGNATURES.items():
    present = [gene for gene in check_genes if gene in panel_genes]
    if len(present) >= MINIMUM_GENES_PER_SIGNATURE:
        SCORED_SIGNATURES[check_name] = present

# Every sheet gene that exists in the panel, for the dotplot later.
ALL_MARKER_GENES = sorted(set(marker_sheet.loc[marker_sheet["in_panel"], "Gene"]))

print()
print(f"Panel: {adata.n_vars} genes. Sheet: {marker_sheet['Gene'].nunique()} unique genes, "
      f"{len(ALL_MARKER_GENES)} of them on the panel.")
print(f"Signatures with >= {MINIMUM_GENES_PER_SIGNATURE} panel genes: {len(SCORED_SIGNATURES)} "
      f"of {len(signature_genes)}")
display(coverage.sort_values("genes_in_panel", ascending=False))

## 3. Subset the lymphocytes

In [ ]:
CELLTYPE_KEY = "manual_celltype"
LYMPHOCYTE_LABEL = "Lymphocytes"

lymphocytes = adata[adata.obs[CELLTYPE_KEY] == LYMPHOCYTE_LABEL].copy()

print("Lymphocytes:", lymphocytes.n_obs)
print()
print(lymphocytes.obs["batch"].value_counts().to_string())
print()
print("Counts per cell (raw):")
print(lymphocytes.obs["transcript_counts"].describe().round(1).to_string())
print()
print("Genes detected per cell:")
print(lymphocytes.obs["n_genes_by_counts"].describe().round(1).to_string())

# Sanity check: do these cells actually look like T/NK cells on the panel?
check_genes = [g for g in ["Cd8a", "Cd4", "Cd3e", "Cd19", "Klrb1", "Ptprc"]
               if g in adata.var_names]
detected_fraction = pd.Series(
    {
        gene: float((lymphocytes[:, gene].X.toarray() > 0).mean())
        for gene in check_genes
    },
    name="fraction_of_lymphocytes_expressing",
)
print()
print("Lineage gene detection within the subset:")
display(detected_fraction.round(3).to_frame())

## 4. Choose a resolution on evidence, not by eye

With a few hundred sparse cells the resolution choice decides the result, so each
candidate is scored on three things: how separated the clusters are (silhouette), how
reproducible they are (ARI across 20 reclustered 80% subsamples), and whether every
cluster is big enough to interpret.

Both gates matter. The finest *reproducible* resolution alone over-splits - a partition
can reshuffle reproducibly and still cut through a continuum - while the best
silhouette alone gives the coarsest, least informative split.

In [ ]:
from sklearn.metrics import silhouette_score, adjusted_rand_score

# X_scVI is already batch-corrected across AD_inf / AD_mock, so neighbours are
# computed on it rather than on a fresh PCA of 429 cells. Retraining scVI on a
# subset this small is not advisable; the trade-off is that a latent space fit
# to all 119,794 cells may under-resolve fine lymphocyte states.
REPRESENTATION = "X_scVI"

# With a few hundred cells the neighbourhood graph is the main lever: keep it
# small enough to see structure, large enough not to chase noise.
N_NEIGHBORS = 10

# A cluster below this size is not interpretable here, so resolutions that
# produce one are not considered when picking a default.
MINIMUM_CLUSTER_SIZE = 15

# Bootstrap stability: recluster repeated 80% subsamples and compare the labels
# back to the full-data labels. Below this mean ARI the extra clusters are
# reshuffling noise rather than reproducible structure.
STABILITY_SUBSAMPLE_FRACTION = 0.8
STABILITY_REPEATS = 20
MINIMUM_STABILITY_ARI = 0.6

# Stability on its own is not enough: a partition can reshuffle reproducibly
# and still cut through a continuum. Require that a resolution also keeps at
# least this share of the best separation seen in the sweep.
SILHOUETTE_RETENTION = 0.5

RESOLUTIONS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0, 1.2, 1.5]


def run_leiden(data, resolution, key, seed=0):
    sc.tl.leiden(
        data, resolution=resolution, key_added=key,
        flavor="igraph", n_iterations=2, directed=False, random_state=seed,
    )
    return data.obs[key]


sc.pp.neighbors(lymphocytes, use_rep=REPRESENTATION, n_neighbors=N_NEIGHBORS)
latent = lymphocytes.obsm[REPRESENTATION]

rng = np.random.default_rng(0)
subsample_size = int(round(STABILITY_SUBSAMPLE_FRACTION * lymphocytes.n_obs))
sweep_rows = []

for resolution in RESOLUTIONS:
    key = f"leiden_sweep_{resolution}"
    labels = run_leiden(lymphocytes, resolution, key)
    cluster_sizes = labels.value_counts()

    # Stability: does the same partition come back from 80% of the cells?
    ari_scores = []
    for repeat in range(STABILITY_REPEATS):
        selected = rng.choice(lymphocytes.n_obs, subsample_size, replace=False)
        subsample = lymphocytes[selected].copy()
        sc.pp.neighbors(subsample, use_rep=REPRESENTATION, n_neighbors=N_NEIGHBORS)
        subsample_labels = run_leiden(subsample, resolution, "leiden_boot", seed=repeat)
        ari_scores.append(
            adjusted_rand_score(labels.iloc[selected].to_numpy(), subsample_labels.to_numpy())
        )

    sweep_rows.append(
        {
            "resolution": resolution,
            "n_clusters": labels.nunique(),
            "smallest_cluster": int(cluster_sizes.min()),
            "largest_cluster": int(cluster_sizes.max()),
            "silhouette": (
                float(silhouette_score(latent, labels)) if labels.nunique() > 1 else np.nan
            ),
            "stability_ari": float(np.mean(ari_scores)),
            "stability_ari_sd": float(np.std(ari_scores)),
        }
    )

sweep = pd.DataFrame(sweep_rows)
silhouette_floor = SILHOUETTE_RETENTION * sweep["silhouette"].max()
sweep["usable"] = (
    (sweep["smallest_cluster"] >= MINIMUM_CLUSTER_SIZE)
    & (sweep["stability_ari"] >= MINIMUM_STABILITY_ARI)
    & (sweep["silhouette"] >= silhouette_floor)
)

# Default: the finest granularity that is both reproducible and still
# separated. Taking the finest reproducible resolution alone over-splits here,
# and taking the best silhouette alone gives the coarsest, least informative
# split, so both gates apply.
usable_sweep = sweep.loc[sweep["usable"]]
SUGGESTED_RESOLUTION = (
    float(usable_sweep["resolution"].max()) if len(usable_sweep) else RESOLUTIONS[0]
)

display(sweep.round(3))
print(f"silhouette floor ({SILHOUETTE_RETENTION:.0%} of best): {silhouette_floor:.3f}")
print("Suggested resolution:", SUGGESTED_RESOLUTION,
      f"-> {int(sweep.loc[sweep.resolution == SUGGESTED_RESOLUTION, 'n_clusters'].iloc[0])} clusters")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(sweep["resolution"], sweep["n_clusters"], marker="o", color="#2a78d6")
axes[0].set_ylabel("clusters")
axes[1].plot(sweep["resolution"], sweep["silhouette"], marker="o", color="#2a78d6")
axes[1].set_ylabel("mean silhouette")
axes[2].errorbar(sweep["resolution"], sweep["stability_ari"],
                 yerr=sweep["stability_ari_sd"], marker="o", color="#2a78d6",
                 ecolor="#9a9992", capsize=3)
axes[2].axhline(MINIMUM_STABILITY_ARI, color="#e34948", linestyle="--", linewidth=1)
axes[2].set_ylabel(f"bootstrap ARI ({STABILITY_REPEATS}x {STABILITY_SUBSAMPLE_FRACTION:.0%})")
for ax in axes:
    ax.axvline(SUGGESTED_RESOLUTION, color="#1baf7a", linestyle=":", linewidth=1.2)
    ax.set_xlabel("leiden resolution")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#e3e2dd", linewidth=0.8)
    ax.set_axisbelow(True)
fig.suptitle(
    f"Leiden resolution sweep, {lymphocytes.n_obs} lymphocytes "
    "(green = chosen, red = stability floor)", y=1.02,
)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "leiden_resolution_sweep.png", dpi=300,
            bbox_inches="tight", facecolor="white")
plt.show()

# The sweep columns are scratch; drop them so obs stays readable.
lymphocytes.obs = lymphocytes.obs.drop(
    columns=[f"leiden_sweep_{resolution}" for resolution in RESOLUTIONS]
)

## 5. Final clustering and UMAP

In [ ]:
# Override this to explore a different granularity; the sweep above is the
# evidence for the default.
RESOLUTION = SUGGESTED_RESOLUTION
LEIDEN_KEY = "lymphocyte_leiden"

run_leiden(lymphocytes, RESOLUTION, LEIDEN_KEY)
sc.tl.umap(lymphocytes, min_dist=0.3, random_state=0)

cluster_composition = pd.crosstab(
    lymphocytes.obs[LEIDEN_KEY], lymphocytes.obs["batch"]
)
cluster_composition["total"] = cluster_composition.sum(axis=1)
cluster_composition["percent_infected"] = (
    100 * cluster_composition.get("AD_inf", 0) / cluster_composition["total"]
)

print(f"Leiden resolution {RESOLUTION}: {lymphocytes.obs[LEIDEN_KEY].nunique()} subclusters")
display(cluster_composition.round(1))

CLUSTER_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
                  "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
cluster_names = lymphocytes.obs[LEIDEN_KEY].cat.categories.tolist()
lymphocytes.uns[f"{LEIDEN_KEY}_colors"] = CLUSTER_COLORS[: len(cluster_names)]

plot_genes = [g for g in ["Cd8a", "Cd3e", "Cd69", "Pdcd1", "Gzmb", "Tcf7", "Isg15", "Mki67"]
              if g in lymphocytes.var_names]

fig = sc.pl.umap(
    lymphocytes, color=[LEIDEN_KEY, "batch"], ncols=2, wspace=0.3,
    frameon=False, size=60, return_fig=True,
)
fig.savefig(FIGURE_DIR / "lymphocyte_umap_clusters.png", dpi=300,
            bbox_inches="tight", facecolor="white")
plt.show()

fig = sc.pl.umap(
    lymphocytes, color=plot_genes, ncols=4, frameon=False, size=60,
    cmap="Blues", return_fig=True,
)
fig.savefig(FIGURE_DIR / "lymphocyte_umap_markers.png", dpi=300,
            bbox_inches="tight", facecolor="white")
plt.show()

## 6. Score the phenotype signatures

Detection depth differs 2.2x between these subclusters, and a cell with more
transcripts scores higher on *every* signature at once. Each cell is therefore
standardised across signatures first, which leaves the relative profile - which
phenotype stands out for this cell. Skipping that step makes the deepest subcluster
top all 20 signatures and the labels meaningless.

In [ ]:
# Score each phenotype signature from the sheet. score_genes compares the
# signature's mean expression against a control set matched for expression
# level, so scores are comparable between signatures of different size.
score_columns = {}
for phenotype, genes in SCORED_SIGNATURES.items():
    column = f"score_{phenotype}"
    sc.tl.score_genes(lymphocytes, gene_list=genes, score_name=column, random_state=0)
    score_columns[phenotype] = column

signature_scores = lymphocytes.obs[list(score_columns.values())].copy()
signature_scores.columns = list(score_columns)

# Detection depth differs a lot between the subclusters here, and a cell with
# more transcripts scores higher on every signature at once. Standardising each
# CELL across signatures removes that uniform shift and leaves the relative
# profile: which phenotype stands out for this cell. Without this step one
# subcluster tops every signature and the labels are meaningless.
relative_scores = signature_scores.sub(signature_scores.mean(axis=1), axis=0).div(
    signature_scores.std(axis=1).replace(0, np.nan), axis=0
)

depth_per_cluster = lymphocytes.obs.groupby(LEIDEN_KEY, observed=True)[
    ["transcript_counts", "n_genes_by_counts", "cell_area"]
].median()
print("Median detection depth per subcluster - check this before trusting labels:")
display(depth_per_cluster.round(1))

# Absolute means, kept for reference, and the relative profile used for labels.
cluster_scores = signature_scores.groupby(lymphocytes.obs[LEIDEN_KEY], observed=True).mean()
cluster_scores_relative = relative_scores.groupby(
    lymphocytes.obs[LEIDEN_KEY], observed=True
).mean()

# Then z-score across subclusters so the heatmap shows which subcluster is
# relatively highest for each signature.
cluster_scores_z = (
    cluster_scores_relative - cluster_scores_relative.mean(axis=0)
).div(cluster_scores_relative.std(axis=0).replace(0, np.nan), axis=1)

# Assign each subcluster the signature it stands out on. This is a label
# suggestion to check against the dotplot, not a classification.
assigned_phenotype = cluster_scores_relative.idxmax(axis=1)
assignment_margin = cluster_scores_relative.apply(
    lambda row: row.nlargest(2).iloc[0] - row.nlargest(2).iloc[1], axis=1
)

assignment = pd.DataFrame(
    {
        "n_cells": lymphocytes.obs[LEIDEN_KEY].value_counts().reindex(cluster_scores.index),
        "median_transcripts": depth_per_cluster["transcript_counts"],
        "top_signature": assigned_phenotype,
        "relative_score": cluster_scores_relative.max(axis=1).round(2),
        "margin_to_second": assignment_margin.round(2),
        "second_signature": cluster_scores_relative.apply(
            lambda row: row.nlargest(2).index[1], axis=1
        ),
    }
)

lymphocytes.obs["lymphocyte_phenotype"] = (
    lymphocytes.obs[LEIDEN_KEY].map(assigned_phenotype).astype("category")
)

print("Relative signature profile per subcluster (mean of per-cell z-scores)")
display(cluster_scores_relative.round(2).T)
print("Suggested label per subcluster")
display(assignment)

# Diverging colour scale: these z-scores have a meaningful zero.
ordered_signatures = cluster_scores_relative.std(axis=0).sort_values(ascending=False).index
matrix = cluster_scores_relative[ordered_signatures].T

fig, ax = plt.subplots(figsize=(3.2 + 1.7 * matrix.shape[1], 0.42 * matrix.shape[0] + 2.0))
limit = np.nanmax(np.abs(matrix.to_numpy()))
image = ax.imshow(matrix, cmap="RdBu_r", vmin=-limit, vmax=limit, aspect="auto")

ax.set_xticks(range(matrix.shape[1]))
ax.set_xticklabels(
    [
        f"{name}\nn={assignment.loc[name, 'n_cells']}\n"
        f"{int(assignment.loc[name, 'median_transcripts'])} tx"
        for name in matrix.columns
    ],
    fontsize=9,
)
ax.set_yticks(range(matrix.shape[0]))
ax.set_yticklabels(matrix.index, fontsize=9)
ax.set_xlabel("lymphocyte subcluster", fontsize=10, color="#52514e")

for row in range(matrix.shape[0]):
    for column in range(matrix.shape[1]):
        value = matrix.iat[row, column]
        if np.isfinite(value):
            ax.text(
                column, row, f"{value:.1f}", ha="center", va="center", fontsize=8,
                color="white" if abs(value) > 0.6 * limit else "#0b0b0b",
            )

ax.set_title(
    "Phenotype signature profile by subcluster\n"
    "per-cell z-score across signatures, averaged per subcluster",
    fontsize=12, loc="left", pad=12,
)
fig.colorbar(image, ax=ax, shrink=0.6, label="relative signature score")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "lymphocyte_signature_heatmap.png", dpi=300,
            bbox_inches="tight", facecolor="white")
plt.show()

## 7. Data-driven markers

Independent of the sheet: what actually separates the subclusters. Where these
disagree with the signature label, trust these.

In [ ]:
# Data-driven markers, independent of the sheet: what actually separates the
# subclusters. Wilcoxon is the right test for counts this sparse.
sc.tl.rank_genes_groups(
    lymphocytes, groupby=LEIDEN_KEY, method="wilcoxon", tie_correct=True,
)

marker_table = sc.get.rank_genes_groups_df(lymphocytes, group=None)
marker_table = marker_table.rename(columns={"group": "subcluster"})
significant_markers = marker_table.loc[
    (marker_table["pvals_adj"] < 0.05) & (marker_table["logfoldchanges"] > 0.5)
]

print("Significant up-markers per subcluster (padj < 0.05, logFC > 0.5):")
print(significant_markers["subcluster"].value_counts().sort_index().to_string())
print()
print("Top 8 per subcluster:")
display(
    significant_markers.groupby("subcluster", observed=True)
    .head(8)[["subcluster", "names", "logfoldchanges", "pvals_adj"]]
    .round(4)
)

# Dotplot of the sheet's genes, grouped by the signatures that vary most
# between subclusters, so the label suggestions can be checked gene by gene.
N_SIGNATURES_TO_SHOW = 8
dotplot_groups = {}
for phenotype in ordered_signatures[:N_SIGNATURES_TO_SHOW]:
    genes = [g for g in SCORED_SIGNATURES[phenotype] if g in lymphocytes.var_names]
    if genes:
        dotplot_groups[phenotype] = genes

fig = sc.pl.dotplot(
    lymphocytes, var_names=dotplot_groups, groupby=LEIDEN_KEY,
    standard_scale="var", cmap="Blues", return_fig=True,
)
fig.savefig(FIGURE_DIR / "lymphocyte_signature_dotplot.png", dpi=300,
            bbox_inches="tight", facecolor="white")
plt.show()

# And the data-driven top markers, for comparison.
top_marker_genes = (
    significant_markers.groupby("subcluster", observed=True).head(6)["names"].unique().tolist()
)
if top_marker_genes:
    fig = sc.pl.dotplot(
        lymphocytes, var_names=top_marker_genes, groupby=LEIDEN_KEY,
        standard_scale="var", cmap="Blues", return_fig=True,
    )
    fig.savefig(FIGURE_DIR / "lymphocyte_de_marker_dotplot.png", dpi=300,
                bbox_inches="tight", facecolor="white")
    plt.show()

## 8. Link the subclusters to plaque proximity

Lymphocytes were the second most plaque-enriched cell type in the earlier analysis.
This asks which subcluster accounts for that.

In [ ]:
# Tie the subclusters back to the plaque-distance work: lymphocytes were the
# second most enriched cell type within 15 µm of a plaque, so which subcluster
# accounts for that?
infected_lymphocytes = lymphocytes.obs.loc[
    lymphocytes.obs["plaque_proximity_valid"].astype(bool)
].copy()

print("Scored (infected) lymphocytes:", len(infected_lymphocytes))

distance_crosstab = pd.crosstab(
    infected_lymphocytes[LEIDEN_KEY],
    infected_lymphocytes["plaque_edge_cutoff_um"],
    dropna=False,
)
print()
print("Subcluster x nearest distance cut-off (exclusive bins, infected only)")
display(distance_crosstab)

cutoff_edges = [float(e) for e in adata.uns["plaque_proximity"]["cutoff_edges_um"]]
proximity_rows = []
for cluster_name, group in infected_lymphocytes.groupby(LEIDEN_KEY, observed=True):
    row = {"subcluster": cluster_name, "n_cells": len(group)}
    for edge in cutoff_edges:
        within = (group["plaque_edge_distance_um"] < edge).sum()
        row[f"percent_within_{edge:g}um"] = 100 * within / len(group)
    row["median_distance_um"] = group["plaque_edge_distance_um"].median()
    proximity_rows.append(row)

subcluster_proximity = pd.DataFrame(proximity_rows).set_index("subcluster")

# Same numbers for all infected cells, as the reference to compare against.
all_infected = adata.obs.loc[adata.obs["plaque_proximity_valid"].astype(bool)]
reference_row = {"n_cells": len(all_infected)}
for edge in cutoff_edges:
    reference_row[f"percent_within_{edge:g}um"] = (
        100 * (all_infected["plaque_edge_distance_um"] < edge).sum() / len(all_infected)
    )
reference_row["median_distance_um"] = all_infected["plaque_edge_distance_um"].median()
subcluster_proximity.loc["all infected cells"] = reference_row

print("Plaque proximity per lymphocyte subcluster")
display(subcluster_proximity.round(1))

fig, ax = plt.subplots(figsize=(8, 4.5))
plot_clusters = [name for name in subcluster_proximity.index if name != "all infected cells"]
bar_width = 0.8 / len(cutoff_edges)
ramp = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#0d366b"]

for position, edge in enumerate(cutoff_edges):
    offset = (position - (len(cutoff_edges) - 1) / 2) * bar_width
    ax.bar(
        np.arange(len(plot_clusters)) + offset,
        subcluster_proximity.loc[plot_clusters, f"percent_within_{edge:g}um"],
        width=bar_width * 0.86, color=ramp[position], label=f"<{edge:g} µm", zorder=3,
    )
    ax.axhline(
        reference_row[f"percent_within_{edge:g}um"], color=ramp[position],
        linestyle=":", linewidth=1, alpha=0.8, zorder=1,
    )

ax.set_xticks(np.arange(len(plot_clusters)))
ax.set_xticklabels(
    [f"{name}\n(n={int(subcluster_proximity.loc[name, 'n_cells'])})" for name in plot_clusters],
    fontsize=9,
)
ax.set_ylabel("% of the subcluster within that distance", fontsize=10, color="#52514e")
ax.set_xlabel("lymphocyte subcluster", fontsize=10, color="#52514e")
ax.set_title(
    "Plaque proximity by lymphocyte subcluster\n"
    "dotted line = all infected cells at that cut-off",
    fontsize=12, loc="left", pad=12,
)
ax.legend(frameon=False, fontsize=9, title="distance", title_fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#e3e2dd", linewidth=0.8)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(FIGURE_DIR / "lymphocyte_subcluster_plaque_proximity.png", dpi=300,
            bbox_inches="tight", facecolor="white")
plt.show()

## 9. Save

In [ ]:
LYMPHOCYTE_H5AD_PATH = OUTPUT_DIR / "lymphocytes_subclustered.h5ad"
PARENT_H5AD_PATH = OUTPUT_DIR / "adata_combined_with_lymphocyte_subclusters.h5ad"
RESULTS_XLSX_PATH = OUTPUT_DIR / "lymphocyte_subclustering_results.xlsx"

lymphocytes.uns["lymphocyte_subclustering"] = {
    "source_h5ad": str(H5AD_PATH),
    "marker_workbook": str(MARKER_XLSX_PATH),
    "marker_sheet": MARKER_SHEET,
    "celltype_key": CELLTYPE_KEY,
    "celltype_label": LYMPHOCYTE_LABEL,
    "representation": REPRESENTATION,
    "n_neighbors": int(N_NEIGHBORS),
    "leiden_resolution": float(RESOLUTION),
    "leiden_flavor": "igraph",
    "n_subclusters": int(lymphocytes.obs[LEIDEN_KEY].nunique()),
    "n_cells": int(lymphocytes.n_obs),
    "stability_ari_at_resolution": float(
        sweep.loc[sweep["resolution"] == RESOLUTION, "stability_ari"].iloc[0]
    ),
    "silhouette_at_resolution": float(
        sweep.loc[sweep["resolution"] == RESOLUTION, "silhouette"].iloc[0]
    ),
    "signatures_scored": list(SCORED_SIGNATURES),
    "created": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
}
lymphocytes.uns["lymphocyte_signature_genes"] = {
    phenotype: np.array(genes, dtype=object)
    for phenotype, genes in SCORED_SIGNATURES.items()
}

lymphocytes.write_h5ad(LYMPHOCYTE_H5AD_PATH, compression="gzip")
print("Wrote", LYMPHOCYTE_H5AD_PATH.name,
      round(LYMPHOCYTE_H5AD_PATH.stat().st_size / 1e6, 1), "MB")

# Carry the labels onto the combined object. Positional, because obs_names can
# repeat between the infected and mock sections.
lymphocyte_positions = np.flatnonzero(
    (adata.obs[CELLTYPE_KEY] == LYMPHOCYTE_LABEL).to_numpy()
)
assert len(lymphocyte_positions) == lymphocytes.n_obs
assert (
    adata.obs_names[lymphocyte_positions] == lymphocytes.obs_names
).all(), "row order mismatch"

for column, dtype_fill in [(LEIDEN_KEY, None), ("lymphocyte_phenotype", None)]:
    values = np.full(adata.n_obs, None, dtype=object)
    values[lymphocyte_positions] = lymphocytes.obs[column].astype(object).to_numpy()
    adata.obs[column] = pd.Categorical(
        values, categories=sorted(lymphocytes.obs[column].astype(str).unique()), ordered=False
    )

for phenotype, column in score_columns.items():
    values = np.full(adata.n_obs, np.nan, dtype="float32")
    values[lymphocyte_positions] = lymphocytes.obs[column].to_numpy()
    adata.obs[column] = values

adata.uns["lymphocyte_subclustering"] = lymphocytes.uns["lymphocyte_subclustering"]
adata.write_h5ad(PARENT_H5AD_PATH, compression="gzip")
print("Wrote", PARENT_H5AD_PATH.name,
      round(PARENT_H5AD_PATH.stat().st_size / 1e6, 1), "MB")


def excel_safe(name):
    """Sheet names: 31 characters, no []:*?/\\ ."""
    for character in "[]:*?/\\":
        name = name.replace(character, "-")
    return name[:31]


with pd.ExcelWriter(RESULTS_XLSX_PATH, engine="openpyxl") as writer:
    sweep.to_excel(writer, sheet_name="resolution_sweep", index=False)
    coverage.to_excel(writer, sheet_name="signature_coverage")
    cluster_composition.to_excel(writer, sheet_name="cluster_composition")
    assignment.to_excel(writer, sheet_name="cluster_labels")
    cluster_scores.round(4).to_excel(writer, sheet_name="signature_scores_mean")
    cluster_scores_z.round(4).to_excel(writer, sheet_name="signature_scores_z")
    marker_table.to_excel(writer, sheet_name="DE_all", index=False)
    significant_markers.to_excel(writer, sheet_name="DE_significant", index=False)
    subcluster_proximity.round(3).to_excel(writer, sheet_name="plaque_proximity")
    distance_crosstab.to_excel(writer, sheet_name="distance_crosstab")
    for cluster_name in lymphocytes.obs[LEIDEN_KEY].cat.categories:
        subset = significant_markers.loc[significant_markers["subcluster"] == cluster_name]
        subset.to_excel(writer, sheet_name=excel_safe(f"cluster_{cluster_name}"), index=False)

print("Wrote", RESULTS_XLSX_PATH.name)
print()
print("Per-cell labels also available as obs columns:",
      [LEIDEN_KEY, "lymphocyte_phenotype"])
print("Figures in:", FIGURE_DIR)

## Caveats

**Only 3 subclusters are supportable.** The sweep is unambiguous: 2-3 clusters are
well separated and highly reproducible, and beyond that silhouette collapses to <0.08
while ARI drifts around 0.6-0.7. Raising `RESOLUTION` will return more clusters, but
they are cuts through a continuum. 429 cells at ~80 genes detected each cannot resolve
the Trm / exhausted / progenitor / effector distinctions the sheet is built for.

**Two of the three subclusters are not CD8 states.** The DE markers are unambiguous:
subcluster 2 is B cells (Ms4a1, Cd19, Ighd, Cd74, H2-Ab1) and subcluster 1 carries
neuronal transcripts (Chgb, Tubb3, Dclk1, Ncam1) at the lowest depth of the three,
which is the signature of ambient RNA or mis-segmentation rather than a lymphocyte
state. Only subcluster 0 is a genuine T-cell population. Consider re-annotating
upstream rather than treating all 429 as lymphocytes.

**The subclustering uses the global scVI latent.** `X_scVI` was fit to all 119,794
cells, so it is batch-corrected but may under-resolve fine lymphocyte structure.
Retraining scVI on 429 cells is not a better option. If you need finer states, the
answer is more cells, not more resolution.

**Signature labels are suggestions.** `margin_to_second` in the `cluster_labels`
sheet says how much to trust each one; a small margin means the top two signatures
are interchangeable.